In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
import os

In [2]:
seed = 3024
print('Random seed: {}'.format(seed))
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

dataset_name = 'PIMA'


num_of_layer = 5
batch_size_train = 32
batch_size_test = 1
number_of_classes = 2



if torch.cuda.is_available():
    print("gpu cuda is available!")
    torch.cuda.manual_seed(seed)
else:
    print("cuda is not available! cpu is available!")
    torch.manual_seed(seed)
    
os.environ["CUDA_VISIBLE_DEVICES"] = '0'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Random seed: 3024
gpu cuda is available!


In [3]:
from torch.utils.data import DataLoader, TensorDataset

file_path = './datasets/PIMA/diabetes.csv'
data = pd.read_csv(file_path, header=0)

X = data.iloc[:, :-1].astype(float).values
y = data.iloc[:, -1].values

indices = np.arange(len(X))
np.random.shuffle(indices)

split = int(0.7 * len(X))
train_indices = indices[:split]
test_indices = indices[split:]

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]

X_train_mean = X_train.mean(axis=0)
X_train_std = X_train.std(axis=0)

X_train = (X_train - X_train_mean) / X_train_std
X_test = (X_test - X_train_mean) / X_train_std

y_train_one_hot = torch.nn.functional.one_hot(torch.tensor(y_train), num_classes=2)
y_test_one_hot = torch.nn.functional.one_hot(torch.tensor(y_test), num_classes=2)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), y_train_one_hot)
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), y_test_one_hot)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

for inputs, labels in train_loader:
    print("Inputs batch shape:", inputs.shape)
    print("Labels batch shape:", labels.shape)
    break

Inputs batch shape: torch.Size([32, 8])
Labels batch shape: torch.Size([32, 2])


In [4]:
y_train.size

537

In [5]:
from cv2 import mean
from sympy import print_rcode


class MergeTemporalDim(nn.Module):
    def __init__(self, T):
        super().__init__()
        self.T = T

    def forward(self, x_seq: torch.Tensor):
        return x_seq.flatten(0, 1).contiguous()

class ExpandTemporalDim(nn.Module):
    def __init__(self, T):
        super().__init__()
        self.T = T

    def forward(self, x_seq: torch.Tensor):
        y_shape = [self.T, int(x_seq.shape[0]/self.T)]
        y_shape.extend(x_seq.shape[1:])
        return x_seq.view(y_shape)

class ZIF(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, gama):
        out = (input >= 0).float()
        L = torch.tensor([gama])
        ctx.save_for_backward(input, out, L)
        return out

    @staticmethod
    def backward(ctx, grad_output):
        (input, out, others) = ctx.saved_tensors
        gama = others[0].item()
        grad_input = grad_output
        tmp = (1 / gama) * (1 / gama) * ((gama - input.abs()).clamp(min=0))
        grad_input = grad_input * tmp
        return grad_input, None

class GradFloor(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        return input.floor()

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output

myfloor = GradFloor.apply

class IF(nn.Module):
    def __init__(self, T=0, L=8, thresh=8.0, tau=1., gama=1.0):
        super(IF, self).__init__()
        self.act = ZIF.apply
        self.thresh = nn.Parameter(torch.tensor([thresh]), requires_grad=True)
        self.tau = tau
        self.gama = gama
        self.expand = ExpandTemporalDim(T)
        self.merge = MergeTemporalDim(T)
        self.L = L
        self.T = T
        self.loss = 0

    def forward(self, x):
        if self.T > 0:
            thre = self.thresh.data
            x = self.expand(x)
            mem = 0.5 * thre
            spike_pot = []
            for t in range(self.T):
                mem = mem + x[t, ...]
                spike = self.act(mem - thre, self.gama) * thre
                mem = mem - spike
                spike_pot.append(spike)
            x = torch.stack(spike_pot, dim=0)
            x = self.merge(x)
        else:
            x = x / self.thresh
            x = torch.clamp(x, 0, 1)
            x = myfloor(x*self.L+0.5)/self.L
            x = x * self.thresh
        return x
    
class MPLayer(nn.Module):
    def __init__(self, T=0, L=8, thresh=8.0, tau=1., gama=1.0):
        super(MPLayer, self).__init__()
        self.act = ZIF.apply
        self.thresh = nn.Parameter(torch.tensor([thresh]), requires_grad=True)
        self.tau = tau
        self.gama = gama
        self.expand = ExpandTemporalDim(T)
        self.merge = MergeTemporalDim(T)
        self.L = L
        self.T = T
        self.loss = 0
        
        self.presim_len = 4
        self.membrane_lower = None
#         self.preex = ExpandTemporalDim(4)

    def forward(self, x):
        
        if self.T > 0:

            thre = self.thresh.data
            x_fake = self.expand(x)
            mem = 0.5 * thre
            spike_pot = []
            for t in range(self.presim_len):
                
                mem = mem + x_fake[t, ...]
                spike = self.act(mem - thre, self.gama) * thre
                mem = mem - spike
                if t == self.presim_len - 1:
                    self.membrane_lower = torch.where(mem>1e-3,torch.ones_like(spike),torch.zeros_like(spike))
#                 spike_pot.append(spike)
#             x_fake = torch.stack(spike_pot, dim=0)
#             x_fake = self.merge(x_fake)
        
        
        if self.T > 0:
            thre = self.thresh.data
            x = self.expand(x)
            mem = 0.5 * thre
            spike_pot = []
            for t in range(self.T):
                mem = mem + x[t, ...]
                spike = self.act(mem - thre, self.gama) * thre
                mem = mem - spike
                spike_pot.append(spike * self.membrane_lower)
            x = torch.stack(spike_pot, dim=0)
            x = self.merge(x)
        else:
            x = x / self.thresh
            x = torch.clamp(x, 0, 1)
            x = myfloor(x*self.L+0.5)/self.L
            x = x * self.thresh
            
        
        return x

def add_dimention(x, T):
    x.unsqueeze_(1)
    x = x.repeat(T, 1, 1, 1, 1)
    return x


In [6]:
import torch
import torch.nn as nn

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        self.T = 0
        self.merge = MergeTemporalDim(0)
        self.expand = ExpandTemporalDim(0)
        
        self.fc1 = nn.Linear(8, 36)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(36, 18)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(18, 10)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(10, 5)
        self.relu4 = nn.ReLU()
        self.fc5 = nn.Linear(5, 3)
        self.relu5 = nn.ReLU()
        self.fc6 = nn.Linear(3, 2)
        
    def set_T(self, T):
        self.T = T
        for module in self.modules():
            if isinstance(module, (IF, ExpandTemporalDim)) or isinstance(module, (MPLayer, ExpandTemporalDim)):
                module.T = T
        return

    def set_L(self, L):
        for module in self.modules():
            if isinstance(module, IF) or isinstance(module, MPLayer):
                module.L = L
        return

    def forward(self, x):
        if self.T > 0:
            x = add_dimention(x, self.T)
            x = self.merge(x)

        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        out = self.relu3(out)
        out = self.fc4(out)
        out = self.relu4(out)
        out = self.fc5(out)
        out = self.relu5(out)
        out = self.fc6(out)
        if self.T > 0:
            out = self.expand(out)
        return out

In [7]:
model = ANN()

model.load_state_dict(torch.load( './model/ANN_' + dataset_name + '.pth'))

model.to(device)

model.set_T(0)
model.set_L(4)

model.eval()
print("Model loaded")

correct = 0
total = 0

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)  
        targets = targets.to(device)  
        
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (predicted == torch.argmax(targets, dim=1)).sum().item()

accuracy = correct / total
print(f'Accuracy: {accuracy:.4f}')

Model loaded
Accuracy: 0.7446


In [8]:
seed = 9989
print('Random seed: {}'.format(seed))
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)



# create a log file
log = open(os.path.join('./log/', dataset_name+'/MLP5_Bayes_' +  f'{seed}'), 'w')
def print_log(print_string, log):
  print("{}".format(print_string))
  log.write('{}\n'.format(print_string))
  log.flush()

Random seed: 9989


In [9]:
import pickle

def save_list_to_file(lst, filename):
    with open(filename, 'wb') as f:
        pickle.dump(lst, f)

def load_list_from_file(filename):
    with open(filename, 'rb') as f:
        lst = pickle.load(f)
    return lst

In [10]:
model.set_T(0)

activation_lists = [[] for _ in range(num_of_layer)]

def hook_fn2(layer_index):
    def fn(module, input, output):
        if output.size(0) == batch_size_train:

            output_max,_ = torch.max(output.view(batch_size_train,-1), dim=1)
            activation_lists[layer_index].append(output_max)

    return fn



hook_list = []
count = 0
for name, module in model.named_children():
    if 'relu' in name:
        
        hook_list += [module.register_forward_hook(hook_fn2(count))]
        count+=1
        
        
# find the maximum activation on training set

layer_labels = []
with torch.no_grad():
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs = inputs.to(device)
        targets.to(device)
        outputs = model(inputs)
        
        layer_labels.append(torch.tensor(targets))


layer_labels = torch.cat(layer_labels, dim=0)

max_active = []
percentile_active = []
for i in range(num_of_layer):
    activation_list = torch.stack(activation_lists[i]).view(-1).detach().cpu().numpy()
    max_active.append(activation_list.max())
    threshold = np.percentile(activation_list, 99.9)
    percentile_active.append(threshold)
    

print_log('threhold for max_active: ',log)

print_log(max_active,log)

print_log('threhold for percentile_active: ',log)

print_log(percentile_active,log)


activation_lists = []
for h in hook_list:
    h.remove()


threhold for max_active: 
[5.0198593, 8.419008, 8.508945, 9.737488, 15.822848]
threhold for percentile_active: 
[4.440101535320319, 8.13081661558153, 8.220849585533161, 9.413323824882529, 15.52806581878664]


C:\Users\10668\AppData\Local\Temp\ipykernel_11708\4012043014.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  layer_labels.append(torch.tensor(targets))


In [11]:
for name, module in model.named_children():
    if 'relu' in name:
        setattr(model, name, IF())

for name, module in model.named_children():
    print(module)



MergeTemporalDim()
ExpandTemporalDim()
Linear(in_features=8, out_features=36, bias=True)
IF(
  (expand): ExpandTemporalDim()
  (merge): MergeTemporalDim()
)
Linear(in_features=36, out_features=18, bias=True)
IF(
  (expand): ExpandTemporalDim()
  (merge): MergeTemporalDim()
)
Linear(in_features=18, out_features=10, bias=True)
IF(
  (expand): ExpandTemporalDim()
  (merge): MergeTemporalDim()
)
Linear(in_features=10, out_features=5, bias=True)
IF(
  (expand): ExpandTemporalDim()
  (merge): MergeTemporalDim()
)
Linear(in_features=5, out_features=3, bias=True)
IF(
  (expand): ExpandTemporalDim()
  (merge): MergeTemporalDim()
)
Linear(in_features=3, out_features=2, bias=True)


In [12]:
with torch.no_grad():
    model.set_T(16)
    model.to(device)
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs).mean(0)
            _, predicted = torch.max(outputs.data, -1)
            total += targets.size(0)
            correct += (predicted.view(-1) == torch.argmax(targets, dim=1)).sum().item()

        
    accuracy = correct / total
    print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.7013


In [13]:
def sample_balanced_batches(original_loader, num_batches=100, target_samples_per_class=None):

    print("Collecting data...")
    
    sample_batch = next(iter(original_loader))
    print(f"Batch structure type: {type(sample_batch)}")
    
    if isinstance(sample_batch, (list, tuple)):
        print(f"Batch is list/tuple, length: {len(sample_batch)}")
        inputs, labels = sample_batch[0], sample_batch[1]
        print(f"Input shape: {inputs.shape}")
        print(f"Label shape: {labels.shape}")
    elif isinstance(sample_batch, dict):
        print(f"Batch is dict, keys: {list(sample_batch.keys())}")
        inputs = sample_batch.get('input', sample_batch.get('x', sample_batch.get('data')))
        labels = sample_batch.get('label', sample_batch.get('y', sample_batch.get('target')))
        print(f"Input shape: {inputs.shape}")
        print(f"Label shape: {labels.shape}")
    else:
        print(f"Batch is tensor, shape: {sample_batch.shape}")
        inputs = sample_batch
        labels = None
    
    all_inputs = []
    all_labels = []
    
    for batch in original_loader:
        if isinstance(batch, (list, tuple)):
            if len(batch) >= 2:
                batch_inputs, batch_labels = batch[0], batch[1]
            else:
                batch_inputs = batch[0]
                batch_labels = None
        elif isinstance(batch, dict):
            batch_inputs = batch.get('input', batch.get('x', batch.get('data')))
            batch_labels = batch.get('label', batch.get('y', batch.get('target')))
        else:
            batch_inputs = batch
            batch_labels = None
        
        all_inputs.append(batch_inputs)
        
        if batch_labels is not None:
            if batch_labels.dim() > 1:
                batch_labels = batch_labels.squeeze()
                if batch_labels.dim() > 1:
                    batch_labels = torch.argmax(batch_labels, dim=1)
            all_labels.append(batch_labels)
        else:
            all_labels.append(torch.zeros(len(batch_inputs), dtype=torch.long))
    
    try:
        all_inputs = torch.cat(all_inputs, dim=0)
        all_labels = torch.cat(all_labels, dim=0)
    except Exception as e:
        print(f"Error merging data: {e}")
        print(f"Inputs shapes: {[x.shape for x in all_inputs]}")
        print(f"Labels shapes: {[x.shape for x in all_labels]}")
        raise
    
    print(f"Original data: {len(all_inputs)} samples")
    print(f"Input shape: {all_inputs.shape}")
    print(f"Label shape: {all_labels.shape}")
    
    if all_labels.dim() > 1:
        print(f"Warning: Label dimension is {all_labels.dim()}, flattening...")
        if all_labels.shape[1] > 1:
            print("Detected one-hot encoding, converting to class indices...")
            all_labels = torch.argmax(all_labels, dim=1)
        else:
            all_labels = all_labels.squeeze()
    
    unique_labels = torch.unique(all_labels)
    num_classes = len(unique_labels)
    
    print(f"Found {num_classes} classes: {unique_labels.tolist()}")
    
    class_data = {}
    for label in unique_labels:
        mask = (all_labels == label)
        
        if mask.dim() > 1:
            mask = mask.squeeze()
        
        class_inputs = all_inputs[mask]
        class_labels = all_labels[mask]
        
        class_data[label.item()] = {
            'inputs': class_inputs,
            'labels': class_labels
        }
        print(f"  Class {label.item()}: {len(class_inputs)} samples")
    
    if target_samples_per_class is None:
        min_class_size = min([len(data['inputs']) for data in class_data.values()])
        samples_per_class = min(min_class_size, 
                              max(50, len(all_inputs) // (num_classes * 2)))
    else:
        samples_per_class = target_samples_per_class
    
    print(f"Samples per class: {samples_per_class}")
    
    sampled_inputs = []
    sampled_labels = []
    
    for label, data in class_data.items():
        inputs = data['inputs']
        labels_data = data['labels']
        
        n_samples = min(samples_per_class, len(inputs))
        
        indices = torch.randperm(len(inputs))[:n_samples]
        sampled_inputs.append(inputs[indices])
        sampled_labels.append(labels_data[indices])
        
        print(f"  Class {label}: sampled {n_samples} samples")
    
    if sampled_inputs:
        sampled_inputs = torch.cat(sampled_inputs, dim=0)
        sampled_labels = torch.cat(sampled_labels, dim=0)
    else:
        print("Warning: No data sampled!")
        return None, (None, None)
    
    print(f"\nBalanced sampling complete: {len(sampled_inputs)} samples")
    print(f"Sampled input shape: {sampled_inputs.shape}")
    print(f"Sampled label shape: {sampled_labels.shape}")
    
    if len(sampled_inputs) > 0:
        shuffle_idx = torch.randperm(len(sampled_inputs))
        sampled_inputs = sampled_inputs[shuffle_idx]
        sampled_labels = sampled_labels[shuffle_idx]
    

    batch_size = 1

    
    dataset = TensorDataset(sampled_inputs, sampled_labels)
    balanced_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )
    
    return balanced_loader, (sampled_inputs, sampled_labels)

def inspect_dataloader(loader, name="DataLoader"):
    print(f"\n{'='*50}")
    print(f"Inspecting {name}")
    print(f"{'='*50}")
    
    batch_count = 0
    for batch in loader:
        batch_count += 1
        print(f"\nBatch {batch_count}:")
        
        if isinstance(batch, (list, tuple)):
            print(f"  Type: list/tuple, length: {len(batch)}")
            for i, item in enumerate(batch):
                print(f"    Item{i}: type={type(item)}, shape={item.shape if hasattr(item, 'shape') else 'N/A'}")
        elif isinstance(batch, dict):
            print(f"  Type: dict, keys: {list(batch.keys())}")
            for key, value in batch.items():
                print(f"    {key}: type={type(value)}, shape={value.shape if hasattr(value, 'shape') else 'N/A'}")
        else:
            print(f"  Type: {type(batch)}, shape={batch.shape if hasattr(batch, 'shape') else 'N/A'}")
        
        if batch_count >= 3:
            break
    
    print(f"\nTotal inspected {batch_count} batches")

inspect_dataloader(train_loader, "Original train_loader")

calib_loader_balanced, calib_data_balanced = sample_balanced_batches(
    train_loader,
    num_batches=100,
    target_samples_per_class=60)

if calib_loader_balanced is not None:
    print("\nBalanced sampling successful!")
    inputs, labels = calib_data_balanced
    print(f"Sampled data shape: inputs={inputs.shape}, labels={labels.shape}")

    unique_labels, counts = torch.unique(labels, return_counts=True)
    for label, count in zip(unique_labels, counts):
        print(f"Class {label.item()}: {count.item()} samples")


Inspecting Original train_loader

Batch 1:
  Type: list/tuple, length: 2
    Item0: type=<class 'torch.Tensor'>, shape=torch.Size([32, 8])
    Item1: type=<class 'torch.Tensor'>, shape=torch.Size([32, 2])

Batch 2:
  Type: list/tuple, length: 2
    Item0: type=<class 'torch.Tensor'>, shape=torch.Size([32, 8])
    Item1: type=<class 'torch.Tensor'>, shape=torch.Size([32, 2])

Batch 3:
  Type: list/tuple, length: 2
    Item0: type=<class 'torch.Tensor'>, shape=torch.Size([32, 8])
    Item1: type=<class 'torch.Tensor'>, shape=torch.Size([32, 2])

Total inspected 3 batches
Batch structure type: <class 'list'>
Batch is list/tuple, length: 2
Input shape: torch.Size([32, 8])
Label shape: torch.Size([32, 2])
Original data: 537 samples
Input shape: torch.Size([537, 8])
Label shape: torch.Size([537])
Found 2 classes: [0, 1]
  Class 0: 348 samples
  Class 1: 189 samples
Samples per class: 60
  Class 0: sampled 60 samples
  Class 1: sampled 60 samples

Balanced sampling complete: 120 samples
Samp

In [14]:

def LS_2(w, M, r_1, r_2):
    L2_up = torch.abs(torch.sum(torch.mm(w, M.t())))
    L2_down = torch.sum(torch.abs(torch.mm(w, M.t())))
    LS_2_value = L2_up / L2_down
    return LS_2_value



def W(new_data_A, new_data_B):
    

    r_1 = new_data_A.shape[0]
    r_2 = new_data_B.shape[0]


    new_data_A = new_data_A.view(r_1,-1).to(device)
    new_data_B = new_data_B.view(r_2,-1).to(device)

    M = torch.zeros((r_1*r_2, new_data_A.shape[1]), dtype=torch.float32)
    M = M.to(device)
    
    index_base = torch.arange(r_2)

    for i in range(r_1):
        M[index_base+i*r_2,:] = new_data_A[i] - new_data_B  

    m = torch.sum(M, axis=0).view(1, -1)
    w = m / torch.norm(m)

    LS_2_value = LS_2(w, M, r_1, r_2).detach().cpu().numpy()
    return LS_2_value


In [15]:
def get_layer_output(target_layer):
    features = []
    hook_handle = None
    
    def _hook_fn(module, input, output):
        features.append(output.detach().cpu())
        
    hook_handle = target_layer.register_forward_hook(_hook_fn)
    
    all_features = []
    all_labels = []
        
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(calib_loader_balanced):

            inputs = inputs.to(device)

            _ = model(inputs)

            if features:                

                batch_features = features.pop()

                if batch_features.dim() > 2:
                    batch_features = batch_features.view(batch_features.size(1), -1)

                all_features.append(batch_features.cpu())
                all_labels.append(labels.cpu())
                

    hook_handle.remove()
    
    if all_features:
        all_features = torch.cat(all_features, dim=0)
        all_labels = torch.cat(all_labels, dim=0)
        
#         print(f"  Total samples: {len(all_features)}")
#         print(f"  Feature dimension: {all_features.shape[1:]}")
#         print(f"  Label distribution: {torch.unique(all_labels, return_counts=True)}")
    else:
        print("Warning: No features extracted")
        all_features = torch.tensor([])
        all_labels = torch.tensor([])
    
    return all_features, all_labels

def compute_multi_ls(features, labels):
    unique_labels = torch.unique(labels)
    class_pairs = []
    LS_1_squence = []
    for class_label in unique_labels:
        class_label = class_label.item()

        pos_mask = (labels == class_label)
        pos_features = features[pos_mask]

        neg_mask = (labels != class_label)
        neg_features = features[neg_mask]

        LS_1_squence.append(W(pos_features,neg_features))
    multi_LS_1 = sum(LS_1_squence)/len(LS_1_squence)
    
    return multi_LS_1

model.set_T(16)
count = 0
for module in model.modules():
    if isinstance(module, IF) or isinstance(module, MPLayer) :
        module.thresh = torch.nn.Parameter(torch.tensor(max_active[count]))
        count += 1

In [16]:
for name, module in model.named_children():
    if 'relu' in name:
        setattr(model, name, nn.ReLU())

for name, module in model.named_children():
    print(module)

LS_ANN = []
for name, module in model.named_children():
    if 'relu' in name:
        layer_features, layer_labels = get_layer_output(module)
        LS_ANN_va = compute_multi_ls(layer_features, layer_labels)
        LS_ANN.append(LS_ANN_va)

print(LS_ANN)


for name, module in model.named_children():
    if 'relu' in name:
        setattr(model, name, IF())
model.to(device)
for name, module in model.named_children():
    print(module)

MergeTemporalDim()
ExpandTemporalDim()
Linear(in_features=8, out_features=36, bias=True)
ReLU()
Linear(in_features=36, out_features=18, bias=True)
ReLU()
Linear(in_features=18, out_features=10, bias=True)
ReLU()
Linear(in_features=10, out_features=5, bias=True)
ReLU()
Linear(in_features=5, out_features=3, bias=True)
ReLU()
Linear(in_features=3, out_features=2, bias=True)
[0.8723002374172211, 0.9202323257923126, 0.991192102432251, 0.9930510222911835, 0.9946953058242798]
MergeTemporalDim()
ExpandTemporalDim()
Linear(in_features=8, out_features=36, bias=True)
IF(
  (expand): ExpandTemporalDim()
  (merge): MergeTemporalDim()
)
Linear(in_features=36, out_features=18, bias=True)
IF(
  (expand): ExpandTemporalDim()
  (merge): MergeTemporalDim()
)
Linear(in_features=18, out_features=10, bias=True)
IF(
  (expand): ExpandTemporalDim()
  (merge): MergeTemporalDim()
)
Linear(in_features=10, out_features=5, bias=True)
IF(
  (expand): ExpandTemporalDim()
  (merge): MergeTemporalDim()
)
Linear(in_fea

In [17]:
from typing import List, Tuple, Dict, Optional
import json
import warnings
warnings.filterwarnings('ignore')

class LayerwiseThresholdOptimizer:
    
    def __init__(
        self,
        model: torch.nn.Module,
        calib_loader,
        max_active_list: List[float],
        layer_names: Optional[List[str]] = None,
        bounds_factor: Tuple[float, float] = (0.7, 1.0),
        n_init: int = 4,
        n_iter: int = 10,
        device: str = None,
        LS_ANN: List[float] = [0.]
    ):
        self.model = model
        self.calib_loader = calib_loader
        self.max_active_list = max_active_list
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.bounds_factor = bounds_factor
        self.n_init = n_init
        self.n_iter = n_iter
        self.LS_ANN = LS_ANN
        
        self.if_layers = []
        self.if_layer_names = []
        
        for name, module in model.named_modules():
            if isinstance(module, IF):
                self.if_layers.append(module)
                self.if_layer_names.append(name)
        
        if len(self.if_layers) != len(max_active_list):
            print(f"Warning: Number of IF layers ({len(self.if_layers)}) does not match max_active list length ({len(max_active_list)})")
            print(f"Using first {min(len(self.if_layers), len(max_active_list))} layers")
        
        self.num_layers = min(len(self.if_layers), len(max_active_list))
        
        if layer_names is not None:
            if len(layer_names) == self.num_layers:
                self.if_layer_names = layer_names
        
        print(f"Layerwise Optimizer Initialization:")
        print(f"  Layers: {self.num_layers}")
        print(f"  Device: {self.device}")
        print(f"  Initial thresholds (max_active): {max_active_list[:self.num_layers]}")
    
    def optimize_layer(self, layer_idx: int) -> Tuple[float, float, Dict]:
        if layer_idx >= self.num_layers:
            raise ValueError(f"Layer index {layer_idx} out of range (0-{self.num_layers-1})")
        
        LS_ANN_layer = self.LS_ANN[layer_idx]
        
        target_layer = self.if_layers[layer_idx]
        layer_name = self.if_layer_names[layer_idx]
        initial_theta = self.max_active_list[layer_idx]
        
        print(f"\n{'='*60}")
        print(f"Optimizing Layer {layer_idx}: {layer_name}")
        print(f"Initial threshold (max_active): {initial_theta:.4f}")
        print(f"{'='*60}")
        
        lower_bound = initial_theta * self.bounds_factor[0]
        upper_bound = initial_theta * self.bounds_factor[1]
        
        print(f"Search range: [{lower_bound:.4f}, {upper_bound:.4f}]")
        print(f"Optimization settings: {self.n_init} initial points + {self.n_iter} Bayesian iterations")
        
        if layer_idx > 0:
            self._freeze_layers_up_to(layer_idx - 1)
        
        for param in target_layer.parameters():
            param.requires_grad = True
        
        optimizer = SingleLayerBOptimizer(
            model=self.model,
            target_layer=target_layer,
            calib_loader=self.calib_loader,
            initial_theta=initial_theta,
            bounds=(lower_bound, upper_bound),
            n_init=self.n_init,
            n_iter=self.n_iter,
            device=self.device,
            LS_ANN_layer = LS_ANN_layer
        )
        
        best_theta, best_score, history = optimizer.optimize()
        
        target_layer.thresh = torch.nn.Parameter(
            torch.tensor(best_theta, dtype=torch.float32, device=self.device)
        )
        
        print(f"\nLayer {layer_idx} optimization complete:")
        print(f"   Optimal threshold: {best_theta:.6f} (initial: {initial_theta:.4f})")
        print(f"   Optimal MultiLS: {best_score:.6f}")
        print(f"   Improvement: {best_score/history['y'][0]:.2f}x")
        
        return best_theta, best_score, history
    
    def optimize_all_layers(self) -> Tuple[List[float], List[float], List[Dict]]:
        print("="*60)
        print("Starting layerwise optimization of all IF layers")
        print("="*60)
        
        optimal_thetas = []
        optimal_scores = []
        all_histories = []
        
        for layer_idx in range(self.num_layers):
            
            if layer_idx < 0:
                optimal_thetas.append(0.8 * self.max_active_list[layer_idx])
                target_layer = self.if_layers[layer_idx]
                target_layer.thresh = torch.nn.Parameter(torch.tensor(optimal_thetas[layer_idx], dtype=torch.float32, device=self.device))
                
                
            else:
                print(f"\n>>> Processing layer {layer_idx+1}/{self.num_layers}")

                best_theta, best_score, history = self.optimize_layer(layer_idx)

                optimal_thetas.append(best_theta)
                optimal_scores.append(best_score)
                all_histories.append(history)

                print(f"\nCurrent progress ({layer_idx+1}/{self.num_layers}):")
        
        print(f"\n{'='*60}")
        print("All layers optimized!")
        print(f"{'='*60}")
        
        
        return optimal_thetas, optimal_scores, all_histories
    
    def _freeze_layers_up_to(self, layer_idx: int):
        for i in range(layer_idx + 1):
            layer = self.if_layers[i]
            for param in layer.parameters():
                param.requires_grad = False
    
   
    def _save_results(self, optimal_thetas: List[float], optimal_scores: List[float], all_histories: List[Dict]):
        import os
        import time
        os.makedirs('./results', exist_ok=True)

        timestamp = time.strftime("%Y%m%d_%H%M%S")

        results = {
            'timestamp': timestamp,
            'optimal_thresholds': optimal_thetas,
            'optimal_scores': optimal_scores,
            'initial_thresholds': self.max_active_list[:self.num_layers],
            'layer_names': self.if_layer_names[:self.num_layers],
            'bounds_factor': self.bounds_factor,
            'n_init': self.n_init,
            'n_iter': self.n_iter,
            'optimization_histories': all_histories
        }

        json_file = f"./results/{dataset_name}/layerwise_threshold_optimization_{seed}.json"
        with open(json_file, 'w') as f:
            json.dump(results, f, indent=2, default=str)
        print(f"\nOptimization results saved to: {json_file}")


from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm
import numpy as np

class SingleLayerBOptimizer:
    
    def __init__(
        self,
        model,
        target_layer,
        calib_loader,
        initial_theta: float,
        bounds: Tuple[float, float],
        n_init: int = 4,
        n_iter: int = 12,
        device: str = 'cpu',
        LS_ANN_layer: float = 0.
    ):
        self.model = model
        self.target_layer = target_layer
        self.calib_loader = calib_loader
        self.initial_theta = initial_theta
        self.bounds = bounds
        self.n_init = n_init
        self.n_iter = n_iter
        self.device = device
        self.LS_ANN_layer = LS_ANN_layer

        self.gp = None
        self.X = []
        self.y = []

    def _evaluate_threshold(self, theta: float) -> float:
        self.target_layer.thresh = torch.nn.Parameter(torch.tensor(theta, device=self.device))
        features, labels = get_layer_output(self.target_layer)
        if len(features) == 0:
            return 1e6
        multi_ls = compute_multi_ls(features, labels)
        return abs(multi_ls - self.LS_ANN_layer)

    def _acquisition(self, theta, xi=0.01):
        theta = np.atleast_2d(theta)
        mu, sigma = self.gp.predict(theta, return_std=True)
        y_best = min(self.y)
        with np.errstate(divide='warn'):
            imp = y_best - mu - xi
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei = np.maximum(ei, 0)
        return ei

    def optimize(self):
        init_err = self._evaluate_threshold(self.initial_theta)
        self.X.append(self.initial_theta)
        self.y.append(init_err)
        best_theta = self.initial_theta
        best_err = init_err
        print(f"Initial point: θ={self.initial_theta:.4f}, error={init_err:.6f}")

        for i in range(self.n_init):
            theta = np.random.uniform(self.bounds[0], self.bounds[1])
            err = self._evaluate_threshold(theta)
            self.X.append(theta)
            self.y.append(err)
            print(f"  Random point {i+1}: θ={theta:.4f}, error={err:.6f}")
            if err < best_err:
                best_err = err
                best_theta = theta
                print(f"    New best!")

        for i in range(self.n_iter):
            kernel = ConstantKernel(1.0) * Matern(length_scale=0.5, nu=2.5) + WhiteKernel(noise_level=0.01)
            self.gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, alpha=1e-6, normalize_y=True)
            X_arr = np.array(self.X).reshape(-1, 1)
            y_arr = np.array(self.y)
            self.gp.fit(X_arr, y_arr)

            res = minimize(lambda x: -self._acquisition(x.reshape(-1,1)),
                           x0=np.random.uniform(self.bounds[0], self.bounds[1]),
                           bounds=[self.bounds], method='L-BFGS-B')
            next_theta = res.x[0]

            err = self._evaluate_threshold(next_theta)
            self.X.append(next_theta)
            self.y.append(err)
            print(f"Iteration {i+1}: θ={next_theta:.4f}, error={err:.6f}")
            if err < best_err:
                best_err = err
                best_theta = next_theta
                print(f"  New best!")

        history = {'X': self.X, 'y': self.y, 'best_theta': best_theta, 'best_score': best_err}
        return best_theta, best_err, history

In [18]:
def test_all_time(th):
    count = 0
    for module in model.modules():
        if isinstance(module, IF) or isinstance(module, MPLayer) :
            module.thresh = torch.nn.Parameter(torch.tensor(th[count]))
            count += 1
            
    sitimes = range(1,32) #[1,2,4,8,16,32]
    
    model.set_T(sitimes[-1])
    model.to(device)
    correct = [0]*len(sitimes)
    total = 0
    model.eval()
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate((test_loader)):
            inputs = inputs.to(device)  
            targets = targets.to(device)  
            
            outputs = model(inputs)
            count1 = 0
            for sitime in sitimes:


                output_mean = outputs[:sitime].mean(0)
                _, predicted = torch.max(output_mean.data, -1)
            
                correct[count1] += (predicted.view(-1) == torch.argmax(targets, dim=1)).sum().item()
                

                count1 += 1

            total += float(targets.size(0))
        final_acc = [100 * zz / total for zz in correct]
        
    return final_acc

In [19]:
if __name__ == "__main__":

    ac_time = 6
    model.set_T(ac_time)
    print("Starting layerwise SNN threshold optimization...")
    
    optimizer = LayerwiseThresholdOptimizer(
        model=model,
        calib_loader=calib_loader_balanced,
        max_active_list=percentile_active,  
        bounds_factor=(0.3, 1.0),
        n_init=20,
        n_iter=50,
        device=device,
        LS_ANN = LS_ANN
    )
    
    LS_active, optimal_scores, histories = optimizer.optimize_all_layers()
    
    print(f"Optimization complete!")
    print(f"Optimal threshold list: {LS_active}")
    


Starting layerwise SNN threshold optimization...
Layerwise Optimizer Initialization:
  Layers: 5
  Device: cuda
  Initial thresholds (max_active): [4.440101535320319, 8.13081661558153, 8.220849585533161, 9.413323824882529, 15.52806581878664]
Starting layerwise optimization of all IF layers

>>> Processing layer 1/5

Optimizing Layer 0: relu1
Initial threshold (max_active): 4.4401
Search range: [1.3320, 4.4401]
Optimization settings: 20 initial points + 50 Bayesian iterations
Initial point: θ=4.4401, error=0.042893
  Random point 1: θ=3.0348, error=0.065516
  Random point 2: θ=3.6980, error=0.055405
  Random point 3: θ=1.6861, error=0.022414
    New best!
  Random point 4: θ=4.1287, error=0.057615
  Random point 5: θ=3.7266, error=0.051572
  Random point 6: θ=2.4986, error=0.039606
  Random point 7: θ=1.8328, error=0.028155
  Random point 8: θ=2.2942, error=0.043729
  Random point 9: θ=3.4748, error=0.060861
  Random point 10: θ=3.9845, error=0.052479
  Random point 11: θ=4.1621, error=

  Random point 17: θ=5.4987, error=0.012275
  Random point 18: θ=3.6866, error=0.013857
  Random point 19: θ=3.0414, error=0.016082
  Random point 20: θ=2.7031, error=0.015315
Iteration 1: θ=3.4759, error=0.016355
Iteration 2: θ=7.7330, error=0.009967
Iteration 3: θ=3.1409, error=0.015119
Iteration 4: θ=6.0798, error=0.002886
  New best!
Iteration 5: θ=3.1514, error=0.015599
Iteration 6: θ=5.2170, error=0.010117
Iteration 7: θ=6.4880, error=0.011912
Iteration 8: θ=6.6388, error=0.008850
Iteration 9: θ=6.0761, error=0.002719
  New best!
Iteration 10: θ=4.7803, error=0.015193
Iteration 11: θ=7.2515, error=0.025093
Iteration 12: θ=8.1582, error=0.008862
Iteration 13: θ=2.7482, error=0.013973
Iteration 14: θ=5.4817, error=0.011784
Iteration 15: θ=3.6957, error=0.015453
Iteration 16: θ=4.4107, error=0.015838
Iteration 17: θ=7.3693, error=0.020609
Iteration 18: θ=2.7425, error=0.014032
Iteration 19: θ=4.5798, error=0.017350
Iteration 20: θ=6.1030, error=0.003424
Iteration 21: θ=4.6824, error

Iteration 33: θ=6.4667, error=0.020249
Iteration 34: θ=9.7721, error=0.031167
Iteration 35: θ=11.9182, error=0.057323
Iteration 36: θ=9.8157, error=0.048644
Iteration 37: θ=9.7037, error=0.032572
Iteration 38: θ=6.7455, error=0.018722
Iteration 39: θ=10.0461, error=0.052809
Iteration 40: θ=15.5182, error=0.060220
Iteration 41: θ=10.2435, error=0.052207
Iteration 42: θ=12.5324, error=0.064917
Iteration 43: θ=6.9648, error=0.016434
Iteration 44: θ=13.7798, error=0.047303
Iteration 45: θ=6.1868, error=0.021631
Iteration 46: θ=10.8192, error=0.054922
Iteration 47: θ=4.9841, error=0.029966
Iteration 48: θ=14.0059, error=0.070123
Iteration 49: θ=14.6266, error=0.048302
Iteration 50: θ=9.7974, error=0.050524

Layer 4 optimization complete:
   Optimal threshold: 7.025674 (initial: 15.5281)
   Optimal MultiLS: 0.014359
   Improvement: 0.24x

Current progress (5/5):

All layers optimized!
Optimization complete!
Optimal threshold list: [1.6451525976764074, 5.760299186183897, 6.076102398671695, 4.

In [20]:

print_log('threhold for max_active: ',log)
print_log(max_active,log)
accs = test_all_time(max_active)
print_log('acc t=1: %.2f t=2: %.2f t=3: %.2f t=4: %.2f t=5: %.2f t=6: %.2f'%(accs[0],accs[1],accs[2],accs[3],accs[4],accs[5]),log)

print_log('threhold for percentile_active: ',log)
print_log(percentile_active,log)    
accs = test_all_time(percentile_active)
print_log('acc t=1: %.2f t=2: %.2f t=3: %.2f t=4: %.2f t=5: %.2f t=6: %.2f'%(accs[0],accs[1],accs[2],accs[3],accs[4],accs[5]),log)

print_log('threhold for LS_active: ',log)
print_log(LS_active,log)    
accs = test_all_time(LS_active)
print_log('acc t=1: %.2f t=2: %.2f t=3: %.2f t=4: %.2f t=5: %.2f t=6: %.2f'%(accs[0],accs[1],accs[2],accs[3],accs[4],accs[5]),log)


threhold for max_active: 
[5.0198593, 8.419008, 8.508945, 9.737488, 15.822848]
acc t=1: 65.80 t=2: 62.34 t=3: 60.17 t=4: 64.50 t=5: 65.80 t=6: 65.80
threhold for percentile_active: 
[4.440101535320319, 8.13081661558153, 8.220849585533161, 9.413323824882529, 15.52806581878664]
acc t=1: 65.80 t=2: 64.07 t=3: 64.50 t=4: 64.07 t=5: 63.64 t=6: 64.94
threhold for LS_active: 
[1.6451525976764074, 5.760299186183897, 6.076102398671695, 4.62128374622206, 7.025673853866792]
acc t=1: 65.80 t=2: 67.97 t=3: 68.83 t=4: 72.73 t=5: 74.89 t=6: 74.89
